<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Using FABRIC NVMe Devices

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook covers:** FABRIC nodes can include dedicated **NVMe (Non-Volatile Memory Express)** storage devices for high-performance I/O workloads. Unlike the built-in local disk, NVMe devices are added as **components** and provide a raw 1 TB block device that you partition, format, and mount yourself. This notebook walks you through reserving and configuring an NVMe device.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Find FABRIC sites that have **NVMe devices available**
2. Add an NVMe component to a node using `add_component()`
3. Inspect the NVMe device details after provisioning
4. **Partition, format, and mount** the NVMe device using `configure_nvme()`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you should:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Successfully run the [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) notebook

**Tip:** NVMe devices are not available at every FABRIC site. This notebook automatically selects a site that has NVMe availability.

</div>

## Background: NVMe Storage on FABRIC

NVMe devices are physical PCI-attached solid-state drives passed through directly to your VM. They offer significantly higher throughput and lower latency than the standard local disk.


**Key facts about NVMe on FABRIC:**
- Model name: `NVME_P4510` (Intel P4510 1 TB)
- The device arrives as a raw PCI block device (e.g., `/dev/nvme0n1`)
- You must partition, format, and mount it before use
- FABlib provides a convenience method `configure_nvme()` that does all three steps
- Data on the NVMe device is **not persistent** -- it is lost when the slice is deleted

## What We're Building

In this notebook we will create a single node with a dedicated NVMe SSD attached via PCI passthrough.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration
fablib.show_config();

---

## Step 2: Select a Site with NVMe Availability

Not all FABRIC sites have NVMe devices. We use `get_random_site()` with a filter function to automatically pick a site that has at least one NVMe device available.

In [ ]:
slice_name="MySlice"

# Select a random site that has at least one NVMe device available
# The filter_function checks the 'nvme_available' column in the site table
site = fablib.get_random_site(filter_function=lambda x: x['nvme_available'] > 0)
print(f"site: {site}")

# Define names for the node and NVMe component
node_name='Node1'
nvme_name='nvme1'

---

## Step 3: Create a Slice with an NVMe Device

NVMe devices are added as **components** using `node.add_component()`. The component model is `NVME_P4510`.

In [ ]:
# Create a new slice
slice = fablib.new_slice(name=slice_name)

# Add a node at the selected site
node = slice.add_node(name=node_name, site=site)

# Add an NVMe drive component to the node
# model='NVME_P4510' specifies the Intel P4510 1TB NVMe SSD
# name='nvme1' is a user-defined label for this component
node.add_component(model='NVME_P4510', name=nvme_name)

# Submit the slice request and wait for provisioning
slice.submit();

---

## Step 4: Inspect the Slice and Node

In [ ]:
# Retrieve and display slice information
slice = fablib.get_slice(name=slice_name)
slice.show();

In [ ]:
# Retrieve the node and display its details
node = slice.get_node(node_name) 
node.show()

# Retrieve and display the NVMe component details
# This shows the PCI device address, model, and status
nvme1 = node.get_component(nvme_name)
nvme1.show();

---

## Step 5: Configure the NVMe Device

The NVMe device arrives as a raw PCI block device. Before you can store files on it, you need to:
1. **Partition** the device
2. **Format** the partition with a filesystem (ext4)
3. **Mount** the filesystem to a directory

The `configure_nvme()` convenience method performs all three steps automatically.

<div class="fab-danger">

**Important:** Running `configure_nvme()` will **erase all data** on the NVMe device. Only run this on a freshly provisioned device or when you intentionally want to wipe the drive.

</div>

In [ ]:
# Partition, format (ext4), and mount the NVMe device
# After this call, the device is mounted and ready to use
nvme1.configure_nvme();

<div class="fab-success">

**What just happened?** The `configure_nvme()` method partitioned the raw NVMe block device, created an ext4 filesystem on it, and mounted it to a directory on the node. You can now read and write files to the mounted NVMe storage.

</div>

---

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. NVMe devices are a limited resource -- leaving them reserved unnecessarily prevents other researchers from using them.

</div>

In [ ]:
# Delete the slice and release all resources (including the NVMe device)
slice = fablib.delete_slice(slice_name)

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `get_random_site()` raises an error | No sites currently have NVMe devices available | Wait and retry later, or check `fablib.list_sites()` for NVMe availability |
| Slice fails during provisioning | The selected site ran out of NVMe devices between selection and provisioning | Retry -- `get_random_site()` will pick a different site |
| `configure_nvme()` fails | Device may not be properly passed through | Check `node.execute('lsblk')` to verify the block device exists |
| Cannot find the mounted directory | `configure_nvme()` may have used a different mount point | Check `node.execute('df -h')` to see where it was mounted |
| Permission denied writing to NVMe mount | Mount point owned by root | Use `sudo` or change ownership with `sudo chown` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_site(filter_function)` | Select a random site matching criteria | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `fablib.delete_slice(name)` | Delete a slice by name | [delete_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.delete_slice) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.get_node(name)` | Get a specific node by name | [get_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_node) |
| `node.add_component(model, name)` | Add a hardware component (NVMe, GPU, etc.) | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.get_component(name)` | Get a component object by name | [get_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_component) |
| `component.configure_nvme()` | Partition, format, and mount the NVMe device | [configure_nvme](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.configure_nvme) |

## What's Next?

Now that you know how to use NVMe storage, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Local Disk** | [local_disk](../local_disk/local_disk.ipynb) | Customize built-in local disk sizes |
| **Persistent Storage** | [persistent_storage](../persistent_storage/persistent_storage.ipynb) | Attach project-level persistent storage that survives slice deletion |
| **GPUs** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Add NVIDIA GPUs to your nodes |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |